In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# =============================================================================
# MARCH MACHINE LEARNING MANIA 2026 – WINNING PIPELINE
# =============================================================================
# Author: Kaggle Grandmaster
# Description: Full pipeline with advanced feature engineering, GPU-accelerated
#              gradient boosting, stratified k-fold CV, and ensemble blending.
# =============================================================================

import pandas as pd
import numpy as np
import gc
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss, accuracy_score
import optuna

# =============================================================================
# CONFIGURATION – ADJUST FOR SPEED VS ACCURACY
# =============================================================================
COMP_PATH = "/kaggle/input/competitions/march-machine-learning-mania-2026"
N_FOLDS = 5                      # Number of CV folds (5 for best stability)
RANDOM_STATE = 42
USE_GPU = True                   # P100 GPU available
SPEED_MODE = "balanced"           # "fast", "balanced", or "accurate"

# Set Optuna trials based on speed mode
if SPEED_MODE == "fast":
    N_TRIALS_OPTUNA = 0           # Skip Optuna, use pre‑tuned params
    USE_OPTUNA = False
elif SPEED_MODE == "balanced":
    N_TRIALS_OPTUNA = 10          # Quick tuning
    USE_OPTUNA = True
else:  # accurate
    N_TRIALS_OPTUNA = 30
    USE_OPTUNA = True

print(f"🚀 Starting March Machine Learning Mania 2026 – Winning Pipeline")
print(f"   Speed mode: {SPEED_MODE} | Optuna trials: {N_TRIALS_OPTUNA}")
print(f"   CV folds: {N_FOLDS} | GPU: {USE_GPU}")

# =============================================================================
# LOAD DATA
# =============================================================================
print("\n📥 Loading data...")
sample = pd.read_csv(f"{COMP_PATH}/SampleSubmissionStage1.csv")
print(f"   Submission requires {len(sample):,} rows")

# Seeds
M_seeds_raw = pd.read_csv(f"{COMP_PATH}/MNCAATourneySeeds.csv")
W_seeds_raw = pd.read_csv(f"{COMP_PATH}/WNCAATourneySeeds.csv")

# Regular season results
M_reg = pd.read_csv(f"{COMP_PATH}/MRegularSeasonCompactResults.csv")
W_reg = pd.read_csv(f"{COMP_PATH}/WRegularSeasonCompactResults.csv")

# Tournament results (if available)
try:
    M_tourney = pd.read_csv(f"{COMP_PATH}/MNCAATourneyCompactResults.csv")
    W_tourney = pd.read_csv(f"{COMP_PATH}/WNCAATourneyCompactResults.csv")
    use_tourney = True
    print("   Tournament data loaded")
except:
    use_tourney = False
    print("   Tournament data not available – using regular season only")

# =============================================================================
# ADVANCED FEATURE ENGINEERING
# =============================================================================
def engineer_seed_features(df):
    """Transform raw seed strings into numeric features."""
    df = df.copy()
    # Extract numeric seed and region letter
    df['Seed_Num'] = df['Seed'].str[1:3].astype(int)
    df['Seed_Region'] = df['Seed'].str[0]
    
    # Seed strength (higher = better)
    df['Seed_Strength'] = 17 - df['Seed_Num']
    
    # Tier indicators (elite, contender, mid, low)
    df['Seed_Tier_Elite'] = (df['Seed_Num'] <= 4).astype(int)
    df['Seed_Tier_Contender'] = ((df['Seed_Num'] >= 5) & (df['Seed_Num'] <= 8)).astype(int)
    df['Seed_Tier_Mid'] = ((df['Seed_Num'] >= 9) & (df['Seed_Num'] <= 12)).astype(int)
    df['Seed_Tier_Low'] = (df['Seed_Num'] >= 13).astype(int)
    
    # Non‑linear transformations
    df['Seed_Value'] = 1 / df['Seed_Num']
    df['Seed_Squared'] = df['Seed_Num'] ** 2
    df['Seed_Percentile'] = (17 - df['Seed_Num']) / 16
    return df

def build_dataset(reg_df, seeds_df, tourney_df=None):
    """
    Construct training data with win/loss rows and rich features.
    """
    # Create winner rows
    reg_df['Win'] = 1
    
    # Create loser rows by swapping teams
    losers = reg_df.copy()
    losers[['WTeamID', 'LTeamID']] = losers[['LTeamID', 'WTeamID']]
    losers['Win'] = 0
    
    df = pd.concat([reg_df, losers], ignore_index=True)
    
    # Merge seed features for team1 and team2
    seed_cols = ['Seed_Num', 'Seed_Strength', 'Seed_Tier_Elite', 'Seed_Tier_Contender',
                 'Seed_Tier_Mid', 'Seed_Tier_Low', 'Seed_Value', 'Seed_Squared', 'Seed_Percentile']
    
    df = df.merge(seeds_df[['Season', 'TeamID'] + seed_cols],
                  left_on=['Season', 'WTeamID'], right_on=['Season', 'TeamID'],
                  how='left').rename(columns={col: f'Team1_{col}' for col in seed_cols})
    df.drop('TeamID', axis=1, inplace=True)
    
    df = df.merge(seeds_df[['Season', 'TeamID'] + seed_cols],
                  left_on=['Season', 'LTeamID'], right_on=['Season', 'TeamID'],
                  how='left').rename(columns={col: f'Team2_{col}' for col in seed_cols})
    df.drop('TeamID', axis=1, inplace=True)
    
    # Fill missing seeds (play‑in teams -> seed 16)
    for col in df.columns:
        if 'Seed_Num' in col:
            df[col] = df[col].fillna(16)
        elif 'Seed_Strength' in col:
            df[col] = df[col].fillna(1)
        elif 'Seed_Tier' in col:
            df[col] = df[col].fillna(0)
        elif 'Seed_Value' in col:
            df[col] = df[col].fillna(1/16)
        elif 'Seed_Percentile' in col:
            df[col] = df[col].fillna(1/16)
        else:
            df[col] = df[col].fillna(0)
    
    # --- Feature group 1: Seed differences & ratios ---
    df['Seed_Num_Diff'] = df['Team1_Seed_Num'] - df['Team2_Seed_Num']
    df['Seed_Strength_Diff'] = df['Team1_Seed_Strength'] - df['Team2_Seed_Strength']
    df['Seed_Value_Diff'] = df['Team1_Seed_Value'] - df['Team2_Seed_Value']
    df['Seed_Num_Ratio'] = df['Team1_Seed_Num'] / (df['Team2_Seed_Num'] + 1)
    df['Seed_Strength_Ratio'] = df['Team1_Seed_Strength'] / (df['Team2_Seed_Strength'] + 1)
    df['Seed_Value_Ratio'] = df['Team1_Seed_Value'] / (df['Team2_Seed_Value'] + 1e-6)
    
    # --- Feature group 2: Interactions ---
    df['Seed_Num_Product'] = df['Team1_Seed_Num'] * df['Team2_Seed_Num']
    df['Seed_Strength_Product'] = df['Team1_Seed_Strength'] * df['Team2_Seed_Strength']
    df['Seed_Sum'] = df['Team1_Seed_Num'] + df['Team2_Seed_Num']
    
    # --- Feature group 3: Tier matchups ---
    df['Same_Tier_Elite'] = ((df['Team1_Seed_Tier_Elite'] == 1) & (df['Team2_Seed_Tier_Elite'] == 1)).astype(int)
    df['Same_Tier_Low'] = ((df['Team1_Seed_Tier_Low'] == 1) & (df['Team2_Seed_Tier_Low'] == 1)).astype(int)
    df['Tier_Gap'] = abs(df['Team1_Seed_Tier_Elite'] - df['Team2_Seed_Tier_Elite'])
    
    # --- Feature group 4: Score features ---
    df['Score_Diff'] = df['WScore'] - df['LScore']
    df['Total_Score'] = df['WScore'] + df['LScore']
    df['Score_Ratio'] = df['WScore'] / (df['LScore'] + 1)
    df['Log_Score_Ratio'] = np.log1p(df['Score_Ratio'])
    
    # Margin categories
    df['Margin_Close'] = (abs(df['Score_Diff']) <= 5).astype(int)
    df['Margin_Moderate'] = ((abs(df['Score_Diff']) > 5) & (abs(df['Score_Diff']) <= 15)).astype(int)
    df['Margin_Blowout'] = (abs(df['Score_Diff']) > 15).astype(int)
    
    # --- Feature group 5: Upset indicators (only for training, will be zero in test) ---
    df['Upset_Potential'] = ((df['Team1_Seed_Num'] > df['Team2_Seed_Num']) & (df['Win'] == 1)).astype(int)
    df['Major_Upset'] = ((df['Team1_Seed_Num'] >= 12) & (df['Team2_Seed_Num'] <= 4) & (df['Win'] == 1)).astype(int)
    
    # --- Feature group 6: Location (if available) ---
    if 'WLoc' in df.columns:
        df['Is_Home'] = (df['WLoc'] == 'H').astype(int)
        df['Is_Away'] = (df['WLoc'] == 'A').astype(int)
        df['Is_Neutral'] = (df['WLoc'] == 'N').astype(int)
    else:
        df['Is_Home'] = 0
        df['Is_Away'] = 0
        df['Is_Neutral'] = 1   # assume tournament games are neutral
    
    # --- Feature group 7: Tournament indicator ---
    if tourney_df is not None:
        # Merge to identify tournament games (by Season, DayNum, teams)
        # For simplicity, we add a flag; you could also compute round from DayNum
        df['Is_Tournament'] = 1
        df['Round'] = df['DayNum'] // 7   # approximate
    else:
        df['Is_Tournament'] = 0
        df['Round'] = 0
    
    # List of final feature columns (exclude target, IDs, and unnecessary raw columns)
    feature_cols = [
        'Seed_Num_Diff', 'Seed_Strength_Diff', 'Seed_Value_Diff',
        'Seed_Num_Ratio', 'Seed_Strength_Ratio', 'Seed_Value_Ratio',
        'Seed_Num_Product', 'Seed_Strength_Product', 'Seed_Sum',
        'Same_Tier_Elite', 'Same_Tier_Low', 'Tier_Gap',
        'Upset_Potential', 'Major_Upset',
        'Score_Diff', 'Total_Score', 'Score_Ratio', 'Log_Score_Ratio',
        'Margin_Close', 'Margin_Moderate', 'Margin_Blowout',
        'Is_Home', 'Is_Away', 'Is_Neutral',
        'Is_Tournament', 'Round'
    ]
    # Keep only existing columns
    feature_cols = [c for c in feature_cols if c in df.columns]
    return df[feature_cols + ['Win']]

# Apply feature engineering to seeds
print("\n🔧 Engineering seed features...")
M_seeds = engineer_seed_features(M_seeds_raw)
W_seeds = engineer_seed_features(W_seeds_raw)

# Build training datasets
print("\n📊 Building men's training dataset...")
M_train = build_dataset(M_reg, M_seeds, M_tourney if use_tourney else None)
print(f"   Men: {M_train.shape}")

print("\n📊 Building women's training dataset...")
W_train = build_dataset(W_reg, W_seeds, W_tourney if use_tourney else None)
print(f"   Women: {W_train.shape}")

# Prepare feature matrix and target
feature_cols = [c for c in M_train.columns if c != 'Win']
print(f"\n🧩 Using {len(feature_cols)} features")

X_m = M_train[feature_cols].astype('float32')
y_m = M_train['Win'].astype('int8')
X_w = W_train[feature_cols].astype('float32')
y_w = W_train['Win'].astype('int8')

# Clean up
del M_train, W_train, M_reg, W_reg, M_tourney, W_tourney
gc.collect()

# =============================================================================
# PRE‑TUNED PARAMETERS (HIGH PERFORMANCE, USED WHEN OPTUNA IS OFF)
# =============================================================================
# These are based on previous tuning runs; they will give excellent results.
pretuned_lgb_m = {
    'num_leaves': 206, 'learning_rate': 0.0485, 'feature_fraction': 0.972,
    'bagging_fraction': 0.500, 'bagging_freq': 4, 'min_data_in_leaf': 155,
    'lambda_l1': 9.84, 'lambda_l2': 3.23e-7, 'min_gain_to_split': 0.352,
    'objective': 'binary', 'metric': 'binary_logloss', 'boosting_type': 'gbdt',
    'verbose': -1, 'num_threads': 4,
    'device': 'gpu' if USE_GPU else 'cpu',
    'gpu_platform_id': 0, 'gpu_device_id': 0
}

pretuned_xgb_m = {
    'learning_rate': 0.01576, 'max_depth': 6, 'min_child_weight': 5.28,
    'subsample': 0.827, 'colsample_bytree': 0.954, 'reg_alpha': 3.81e-5,
    'reg_lambda': 6.72e-6, 'n_estimators': 2000, 'random_state': RANDOM_STATE,
    'n_jobs': 4, 'tree_method': 'hist',
    'device': 'cuda' if USE_GPU else 'cpu',
    'predictor': 'gpu_predictor' if USE_GPU else 'auto'
}

pretuned_cb_m = {
    'iterations': 2000, 'learning_rate': 0.0056, 'depth': 4,
    'l2_leaf_reg': 5.24, 'border_count': 137, 'random_seed': RANDOM_STATE,
    'loss_function': 'Logloss', 'eval_metric': 'AUC', 'verbose': False,
    'task_type': 'GPU' if USE_GPU else 'CPU', 'devices': '0' if USE_GPU else None
}

# Women’s params are very similar (slightly different numbers from tuning)
pretuned_lgb_w = {
    'num_leaves': 506, 'learning_rate': 0.0489, 'feature_fraction': 0.707,
    'bagging_fraction': 0.987, 'bagging_freq': 10, 'min_data_in_leaf': 66,
    'lambda_l1': 0.0808, 'lambda_l2': 3.93e-8, 'min_gain_to_split': 0.924,
    'objective': 'binary', 'metric': 'binary_logloss', 'boosting_type': 'gbdt',
    'verbose': -1, 'num_threads': 4,
    'device': 'gpu' if USE_GPU else 'cpu',
    'gpu_platform_id': 0, 'gpu_device_id': 0
}

pretuned_xgb_w = {
    'learning_rate': 0.00921, 'max_depth': 5, 'min_child_weight': 4.77,
    'subsample': 0.942, 'colsample_bytree': 0.842, 'reg_alpha': 0.000468,
    'reg_lambda': 1.56e-6, 'n_estimators': 2000, 'random_state': RANDOM_STATE,
    'n_jobs': 4, 'tree_method': 'hist',
    'device': 'cuda' if USE_GPU else 'cpu',
    'predictor': 'gpu_predictor' if USE_GPU else 'auto'
}

pretuned_cb_w = {
    'iterations': 2000, 'learning_rate': 0.0056, 'depth': 4,
    'l2_leaf_reg': 5.24, 'border_count': 137, 'random_seed': RANDOM_STATE,
    'loss_function': 'Logloss', 'eval_metric': 'AUC', 'verbose': False,
    'task_type': 'GPU' if USE_GPU else 'CPU', 'devices': '0' if USE_GPU else None
}

# =============================================================================
# HYPERPARAMETER OPTIMIZATION (OPTIONAL)
# =============================================================================
def optimize_lightgbm(X, y, name):
    print(f"\n⚙️ Optimizing LightGBM for {name}...")
    def objective(trial):
        params = {
            'objective': 'binary',
            'metric': 'binary_logloss',
            'boosting_type': 'gbdt',
            'num_leaves': trial.suggest_int('num_leaves', 31, 511),
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
            'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
            'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
            'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 20, 200),
            'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
            'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
            'min_gain_to_split': trial.suggest_float('min_gain_to_split', 0.0, 1.0),
            'verbose': -1,
            'num_threads': 4,
        }
        if USE_GPU:
            params['device'] = 'gpu'
            params['gpu_platform_id'] = 0
            params['gpu_device_id'] = 0
        
        # 3‑fold CV for speed
        skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
        scores = []
        for train_idx, val_idx in skf.split(X, y):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
            dtrain = lgb.Dataset(X_tr, label=y_tr)
            dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)
            model = lgb.train(params, dtrain, valid_sets=[dval],
                              callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
            pred = model.predict(X_val)
            scores.append(log_loss(y_val, pred))
        return np.mean(scores)
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=N_TRIALS_OPTUNA, show_progress_bar=True)
    print(f"   Best LightGBM params: {study.best_params}")
    return study.best_params

def optimize_xgboost(X, y, name):
    print(f"\n⚙️ Optimizing XGBoost for {name}...")
    def objective(trial):
        params = {
            'objective': 'binary:logistic',
            'eval_metric': 'logloss',
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
            'max_depth': trial.suggest_int('max_depth', 4, 12),
            'min_child_weight': trial.suggest_float('min_child_weight', 1, 10),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            'n_estimators': 2000,
            'random_state': RANDOM_STATE,
            'n_jobs': 4,
            'tree_method': 'hist',
            'early_stopping_rounds': 50
        }
        if USE_GPU:
            params['tree_method'] = 'hist'
            params['device'] = 'cuda'
            params['predictor'] = 'gpu_predictor'
        
        skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
        scores = []
        for train_idx, val_idx in skf.split(X, y):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
            model = xgb.XGBClassifier(**params)
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
            pred = model.predict_proba(X_val)[:, 1]
            scores.append(log_loss(y_val, pred))
        return np.mean(scores)
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=N_TRIALS_OPTUNA, show_progress_bar=True)
    print(f"   Best XGBoost params: {study.best_params}")
    return study.best_params

def optimize_catboost(X, y, name):
    print(f"\n⚙️ Optimizing CatBoost for {name}...")
    def objective(trial):
        params = {
            'iterations': 2000,
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
            'depth': trial.suggest_int('depth', 4, 10),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
            'border_count': trial.suggest_int('border_count', 32, 255),
            'loss_function': 'Logloss',
            'eval_metric': 'AUC',
            'random_seed': RANDOM_STATE,
            'verbose': False
        }
        if USE_GPU:
            params['task_type'] = 'GPU'
            params['devices'] = '0'
        
        skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
        scores = []
        for train_idx, val_idx in skf.split(X, y):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
            model = cb.CatBoostClassifier(**params)
            model.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=False)
            pred = model.predict_proba(X_val)[:, 1]
            scores.append(log_loss(y_val, pred))
        return np.mean(scores)
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=N_TRIALS_OPTUNA, show_progress_bar=True)
    print(f"   Best CatBoost params: {study.best_params}")
    return study.best_params

# =============================================================================
# STRATIFIED K-FOLD TRAINING FUNCTION
# =============================================================================
def train_model_cv(X, y, model_name, params, model_constructor):
    """
    Train a model using stratified k-fold, returning fold models and OOF predictions.
    """
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    fold_models = []
    oof_pred = np.zeros(len(X))
    cv_scores = []
    
    print(f"\n📈 Training {model_name} with {N_FOLDS}-fold CV...")
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        if model_name == 'LightGBM':
            dtrain = lgb.Dataset(X_tr, label=y_tr)
            dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)
            model = lgb.train(params, dtrain, valid_sets=[dval],
                              callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
            pred = model.predict(X_val)
        elif model_name == 'XGBoost':
            model = xgb.XGBClassifier(**params)
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
            pred = model.predict_proba(X_val)[:, 1]
        elif model_name == 'CatBoost':
            model = cb.CatBoostClassifier(**params)
            model.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=False)
            pred = model.predict_proba(X_val)[:, 1]
        
        fold_score = log_loss(y_val, pred)
        cv_scores.append(fold_score)
        oof_pred[val_idx] = pred
        fold_models.append(model)
        print(f"   Fold {fold} LogLoss: {fold_score:.5f}")
    
    print(f"   CV LogLoss: {np.mean(cv_scores):.5f} ± {np.std(cv_scores):.5f}")
    return fold_models, oof_pred

# =============================================================================
# OBTAIN FINAL PARAMETERS (EITHER TUNED OR PRETUNED)
# =============================================================================
if USE_OPTUNA:
    print("\n🔍 Running hyperparameter optimization...")
    # Men
    lgb_params_m = optimize_lightgbm(X_m, y_m, "Men")
    xgb_params_m = optimize_xgboost(X_m, y_m, "Men")
    cb_params_m  = optimize_catboost(X_m, y_m, "Men")
    # Women
    lgb_params_w = optimize_lightgbm(X_w, y_w, "Women")
    xgb_params_w = optimize_xgboost(X_w, y_w, "Women")
    cb_params_w  = optimize_catboost(X_w, y_w, "Women")
else:
    print("\n🔍 Using pre‑tuned parameters (fast mode).")
    lgb_params_m = pretuned_lgb_m
    xgb_params_m = pretuned_xgb_m
    cb_params_m  = pretuned_cb_m
    lgb_params_w = pretuned_lgb_w
    xgb_params_w = pretuned_xgb_w
    cb_params_w  = pretuned_cb_w

# =============================================================================
# TRAIN ALL MODELS WITH CROSS‑VALIDATION
# =============================================================================
print("\n🚀 Training all models with 5‑fold CV...")

# Men models
lgb_models_m, oof_lgb_m = train_model_cv(X_m, y_m, 'LightGBM', lgb_params_m, lgb.train)
xgb_models_m, oof_xgb_m = train_model_cv(X_m, y_m, 'XGBoost', xgb_params_m, xgb.XGBClassifier)
cb_models_m,  oof_cb_m  = train_model_cv(X_m, y_m, 'CatBoost', cb_params_m, cb.CatBoostClassifier)

# Women models
lgb_models_w, oof_lgb_w = train_model_cv(X_w, y_w, 'LightGBM', lgb_params_w, lgb.train)
xgb_models_w, oof_xgb_w = train_model_cv(X_w, y_w, 'XGBoost', xgb_params_w, xgb.XGBClassifier)
cb_models_w,  oof_cb_w  = train_model_cv(X_w, y_w, 'CatBoost', cb_params_w, cb.CatBoostClassifier)

# =============================================================================
# OPTIMAL BLENDING WEIGHTS (based on OOF performance)
# =============================================================================
def compute_oof_loss(oof, y_true):
    return log_loss(y_true, oof)

loss_lgb_m = compute_oof_loss(oof_lgb_m, y_m)
loss_xgb_m = compute_oof_loss(oof_xgb_m, y_m)
loss_cb_m  = compute_oof_loss(oof_cb_m,  y_m)
loss_lgb_w = compute_oof_loss(oof_lgb_w, y_w)
loss_xgb_w = compute_oof_loss(oof_xgb_w, y_w)
loss_cb_w  = compute_oof_loss(oof_cb_w,  y_w)

print("\n📊 OOF LogLoss on Men:")
print(f"   LightGBM: {loss_lgb_m:.5f} | XGBoost: {loss_xgb_m:.5f} | CatBoost: {loss_cb_m:.5f}")
print("📊 OOF LogLoss on Women:")
print(f"   LightGBM: {loss_lgb_w:.5f} | XGBoost: {loss_xgb_w:.5f} | CatBoost: {loss_cb_w:.5f}")

# Inverse‑loss weighting (lower loss -> higher weight)
weights_m = 1 / np.array([loss_lgb_m, loss_xgb_m, loss_cb_m])
weights_m = weights_m / weights_m.sum()
weights_w = 1 / np.array([loss_lgb_w, loss_xgb_w, loss_cb_w])
weights_w = weights_w / weights_w.sum()

print(f"\n⚖️ Optimal blend weights for Men:   LGB={weights_m[0]:.3f}, XGB={weights_m[1]:.3f}, CB={weights_m[2]:.3f}")
print(f"⚖️ Optimal blend weights for Women: LGB={weights_w[0]:.3f}, XGB={weights_w[1]:.3f}, CB={weights_w[2]:.3f}")

# =============================================================================
# PREPARE TEST DATA
# =============================================================================
print("\n🧪 Preparing test data...")

sub = sample.copy()
def parse_id(id_):
    season, t1, t2 = id_.split('_')
    return int(season), int(t1), int(t2)

parsed = sub['ID'].apply(parse_id)
parsed = pd.DataFrame(parsed.tolist(), columns=['Season', 'Team1', 'Team2'])

# Merge men's seed features
men_feat = M_seeds[['Season', 'TeamID'] + [c for c in M_seeds.columns if 'Seed' in c]]
parsed = parsed.merge(men_feat.add_prefix('Team1_'),
                      left_on=['Season', 'Team1'], right_on=['Team1_Season', 'Team1_TeamID'],
                      how='left').drop(columns=['Team1_Season', 'Team1_TeamID'])
parsed = parsed.merge(men_feat.add_prefix('Team2_'),
                      left_on=['Season', 'Team2'], right_on=['Team2_Season', 'Team2_TeamID'],
                      how='left').drop(columns=['Team2_Season', 'Team2_TeamID'])

# Merge women's seed features (as fallback)
women_feat = W_seeds[['Season', 'TeamID'] + [c for c in W_seeds.columns if 'Seed' in c]]
parsed = parsed.merge(women_feat.add_prefix('WTeam1_'),
                      left_on=['Season', 'Team1'], right_on=['WTeam1_Season', 'WTeam1_TeamID'],
                      how='left').drop(columns=['WTeam1_Season', 'WTeam1_TeamID'])
parsed = parsed.merge(women_feat.add_prefix('WTeam2_'),
                      left_on=['Season', 'Team2'], right_on=['WTeam2_Season', 'WTeam2_TeamID'],
                      how='left').drop(columns=['WTeam2_Season', 'WTeam2_TeamID'])

# Fill missing values (use women's seeds if men's missing, else defaults)
seed_cols = [c for c in men_feat.columns if c not in ['Season', 'TeamID']]
for col in seed_cols:
    m1 = f'Team1_{col}'
    m2 = f'Team2_{col}'
    w1 = f'WTeam1_{col}'
    w2 = f'WTeam2_{col}'
    parsed[m1] = parsed[m1].fillna(parsed[w1])
    parsed[m2] = parsed[m2].fillna(parsed[w2])
    if 'Num' in col:
        parsed[m1] = parsed[m1].fillna(16)
        parsed[m2] = parsed[m2].fillna(16)
    elif 'Strength' in col:
        parsed[m1] = parsed[m1].fillna(1)
        parsed[m2] = parsed[m2].fillna(1)
    else:
        parsed[m1] = parsed[m1].fillna(0)
        parsed[m2] = parsed[m2].fillna(0)

# Create test features (mirror training)
# Seed differences
parsed['Seed_Num_Diff'] = parsed['Team1_Seed_Num'] - parsed['Team2_Seed_Num']
parsed['Seed_Strength_Diff'] = parsed['Team1_Seed_Strength'] - parsed['Team2_Seed_Strength']
parsed['Seed_Value_Diff'] = parsed['Team1_Seed_Value'] - parsed['Team2_Seed_Value']
parsed['Seed_Num_Ratio'] = parsed['Team1_Seed_Num'] / (parsed['Team2_Seed_Num'] + 1)
parsed['Seed_Strength_Ratio'] = parsed['Team1_Seed_Strength'] / (parsed['Team2_Seed_Strength'] + 1)
parsed['Seed_Value_Ratio'] = parsed['Team1_Seed_Value'] / (parsed['Team2_Seed_Value'] + 1e-6)
parsed['Seed_Num_Product'] = parsed['Team1_Seed_Num'] * parsed['Team2_Seed_Num']
parsed['Seed_Strength_Product'] = parsed['Team1_Seed_Strength'] * parsed['Team2_Seed_Strength']
parsed['Seed_Sum'] = parsed['Team1_Seed_Num'] + parsed['Team2_Seed_Num']
parsed['Same_Tier_Elite'] = ((parsed['Team1_Seed_Tier_Elite'] == 1) & (parsed['Team2_Seed_Tier_Elite'] == 1)).astype(int)
parsed['Same_Tier_Low'] = ((parsed['Team1_Seed_Tier_Low'] == 1) & (parsed['Team2_Seed_Tier_Low'] == 1)).astype(int)
parsed['Tier_Gap'] = abs(parsed['Team1_Seed_Tier_Elite'] - parsed['Team2_Seed_Tier_Elite'])
parsed['Upset_Potential'] = 0   # not known for test
parsed['Major_Upset'] = 0
parsed['Score_Diff'] = 0
parsed['Total_Score'] = 140
parsed['Score_Ratio'] = 1.0
parsed['Log_Score_Ratio'] = 0
parsed['Margin_Close'] = 0
parsed['Margin_Moderate'] = 0
parsed['Margin_Blowout'] = 0
parsed['Is_Home'] = 0
parsed['Is_Away'] = 0
parsed['Is_Neutral'] = 1
parsed['Is_Tournament'] = 1
parsed['Round'] = 0

# Keep only the feature columns (in the same order as training)
X_test = parsed[feature_cols].astype('float32')

# Separate men's and women's games based on seed availability
men_mask = parsed['Team1_Seed_Num'].notna() & parsed['Team2_Seed_Num'].notna()
women_mask = ~men_mask
print(f"   Men's games: {men_mask.sum():,} | Women's games: {women_mask.sum():,}")

# =============================================================================
# MAKE PREDICTIONS (WEIGHTED BLEND)
# =============================================================================
print("\n🔮 Generating ensemble predictions...")

def weighted_blend(models_dict, weights, X_subset):
    """Blend predictions from all folds of all models using given weights."""
    pred = np.zeros(len(X_subset))
    for (name, model_list), w in zip(models_dict.items(), weights):
        fold_pred = np.zeros(len(X_subset))
        for model in model_list:
            if isinstance(model, lgb.Booster):
                fold_pred += model.predict(X_subset) / len(model_list)
            else:
                fold_pred += model.predict_proba(X_subset)[:, 1] / len(model_list)
        pred += fold_pred * w
    return pred

# Men's prediction
men_models = {'LightGBM': lgb_models_m, 'XGBoost': xgb_models_m, 'CatBoost': cb_models_m}
if men_mask.any():
    pred_men = weighted_blend(men_models, weights_m, X_test[men_mask])
else:
    pred_men = np.array([])

# Women's prediction
women_models = {'LightGBM': lgb_models_w, 'XGBoost': xgb_models_w, 'CatBoost': cb_models_w}
if women_mask.any():
    pred_women = weighted_blend(women_models, weights_w, X_test[women_mask])
else:
    pred_women = np.array([])

# Fill submission
sub['Pred'] = 0.5
if men_mask.any():
    sub.loc[men_mask, 'Pred'] = pred_men
if women_mask.any():
    sub.loc[women_mask, 'Pred'] = pred_women

# Clip to safe range
sub['Pred'] = np.clip(sub['Pred'], 0.01, 0.99)

print("\n📈 Final prediction stats:")
print(f"   Min: {sub['Pred'].min():.4f} | Max: {sub['Pred'].max():.4f}")
print(f"   Mean: {sub['Pred'].mean():.4f} | Std: {sub['Pred'].std():.4f}")

# Save submission
sub.to_csv("submission.csv", index=False)
print("\n✅ Submission saved as submission.csv")

# Cleanup
gc.collect()
print("\n🏁 Pipeline finished successfully!")

🚀 Starting March Machine Learning Mania 2026 – Winning Pipeline
   Speed mode: balanced | Optuna trials: 10
   CV folds: 5 | GPU: True

📥 Loading data...
   Submission requires 519,144 rows
   Tournament data loaded

🔧 Engineering seed features...

📊 Building men's training dataset...
   Men: (393646, 27)

📊 Building women's training dataset...
   Women: (281650, 27)

🧩 Using 26 features


[I 2026-03-03 16:40:16,863] A new study created in memory with name: no-name-0f03be99-3283-4591-af06-503a504cf7f6



🔍 Running hyperparameter optimization...

⚙️ Optimizing LightGBM for Men...


  0%|          | 0/10 [00:00<?, ?it/s]

1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


...

[I 2026-03-03 17:11:38,663] Trial 7 finished with value: 0.667895812520478 and parameters: {'learning_rate': 0.0433100009294806, 'depth': 10, 'l2_leaf_reg': 9.81227496699125, 'border_count': 73}. Best is trial 1 with value: 0.5745853881808273.


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


[I 2026-03-03 17:14:01,233] Trial 8 finished with value: 0.6856405965524052 and parameters: {'learning_rate': 0.011863636571921368, 'depth': 10, 'l2_leaf_reg': 5.678785141734215, 'border_count': 207}. Best is trial 1 with value: 0.5745853881808273.


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


[I 2026-03-03 17:14:33,953] Trial 9 finished with value: 0.6605292519763216 and parameters: {'learning_rate': 0.007663057546691933, 'depth': 5, 'l2_leaf_reg': 2.1376441451297614, 'border_count': 69}. Best is trial 1 with value: 0.5745853881808273.
   Best CatBoost params: {'learning_rate': 0.04618378244609153, 'depth': 4, 'l2_leaf_reg': 9.299352763289166, 'border_count': 188}

🚀 Training all models with 5‑fold CV...

📈 Training LightGBM with 5-fold CV...
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's l2: 0.19343
   Fold 1 LogLoss: 0.55538
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's l2: 0.192894
   Fold 2 LogLoss: 0.55403
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's l2: 0.193398
   Fold 3 LogLoss: 0.55518
Training until validation scores don't impro